In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import time
import numpy as np
import pickle
from copy import deepcopy

from naludaq.board import Board
from naludaq.communication import AnalogRegisters, DigitalRegisters, ControlRegisters
from naludaq.parsers import get_parser
from naludaq.tools import get_pedestals_controller
from naludaq.tools.pedestals.pedestals_correcter import PedestalsCorrecter

from naludaq.backend import AcquisitionManager
from naludaq.controllers import (
    get_board_controller,
    get_dac_controller,
    get_readout_controller,
)

from naluinstruments.Instruments import Siglent_FuncGen

import matplotlib.pyplot as plt

DATA_DIR = Path("./data").absolute()
DATA_DIR.mkdir(exist_ok=True)
SERVER_DIR = Path("./server").absolute()
SERVER_DIR.mkdir(exist_ok=True)
FIGURES_DIR = Path("figures").absolute()
FIGURES_DIR.mkdir(exist_ok=True)

In [ ]:
# WLC Supported chips: asocv3, hdsocv1, hdsocv2
BOARD = Board("hdsocv2_evalr2")
BOARD.start_server("./server")
BOARD_ADDRESS = ("192.168.1.60", 4660)
RECEIVER_ADDRESS = ("192.168.1.165", 4665)
SERVER_ADDRESS = (BOARD.context.address)
BOARD.connect_udp(BOARD_ADDRESS, RECEIVER_ADDRESS, SERVER_ADDRESS)
BOARD.initialize()

### Function Definitions

In [ ]:
def create_acquisition(name: str, board: Board, readout_metadata: dict = None):
    am = AcquisitionManager(board)
    acquisition = am.create(name)
    acquisition.set_output()
    if readout_metadata:
        acquisition.readout_metadata = readout_metadata
    return acquisition

def parse_data(raw_data: dict, board: Board, correct_peds: bool = True):
    PARSER = get_parser(board.params)
    evt = PARSER.parse(raw_data)
    if correct_peds:
        PC = PedestalsCorrecter(board.params, board.pedestals)
        PC.run(evt)
    return evt

def create_acq_name(base_name: str):
    timestr = time.strftime("%Y-%m-%d-%H-%M-%S")
    return base_name + "_" + timestr

def clear_figs():
    plt.close("all")

def create_fig(figsize: tuple):
    fig = plt.figure(figsize=figsize, constrained_layout=True)
    fig.set_facecolor('white')
    return fig

def plot_single_event(event, show: bool = True):
    for chan, chan_data in enumerate(event["data"]):
        if isinstance(chan_data, np.ndarray) and chan_data.size == 0:
            continue
        if len(chan_data) == 0:
            continue
        ax = plt.plot(chan_data, '.-', markersize=10, label=f"Channel {chan}")
    plt.legend(fontsize=12)
    if show:
        plt.show()


### Function Generator Setup

In [ ]:
PULSE_WIDTH = 50e-9
PULSE_FREQ = 1 / (5e-6)
N_CYCLES = 3
BURST_PERIOD = 0.5

CHANNEL_INPUT = Siglent_FuncGen(1, "192.168.1.124")
TRIGGER_INPUT = Siglent_FuncGen(2, "192.168.1.124")
CHANNEL_INPUT.rf_off()
TRIGGER_INPUT.rf_off()

# 11 MHz sine wave, 100 mV amplitude, 50 ohm load, no burst
CHANNEL_INPUT.set_burst("OFF")
CHANNEL_INPUT.set_load("50")
CHANNEL_INPUT.set_wavetype("SINE")
CHANNEL_INPUT.set_offset(0)
CHANNEL_INPUT.set_freq(11e6)
CHANNEL_INPUT.set_amp(0.1)

# 5 microsecond pulse period, 50 ns wide, 2.5 V amplitude pulses,
# 50 ohm load, burst of 7 pulses with 1 second period bursts
TRIGGER_INPUT.set_burst("OFF")
TRIGGER_INPUT.set_load("50")
TRIGGER_INPUT.set_wavetype("PULSE")
TRIGGER_INPUT.set_offset(0)
TRIGGER_INPUT.set_freq(PULSE_FREQ) # 5 microseconds between pulses
TRIGGER_INPUT.set_amp(2.5)
TRIGGER_INPUT.set_width(PULSE_WIDTH) # 50 ns wide pulses
TRIGGER_INPUT.set_burst("ON")
TRIGGER_INPUT.set_burst_mode("NCYC")
TRIGGER_INPUT.set_burst_time(N_CYCLES)
TRIGGER_INPUT.set_burst_period(BURST_PERIOD)
TRIGGER_INPUT.set_delay(0)
TRIGGER_INPUT.set_burst_triggersource("INT")

print(CHANNEL_INPUT.get_output_settings())
print(CHANNEL_INPUT.get_burst_settings())
print(TRIGGER_INPUT.get_output_settings())
print(TRIGGER_INPUT.get_burst_settings())

### Setup and Readout Data

In [ ]:
TRIGGER_TIME = 20
WINDOWS = 8
LOOKBACK = WINDOWS + 2
WAT = 0
DAC_LEVEL = 1804
READOUT_CHANNELS = [0, 1, 2, 3]
WLC_ON = 1

AR = AnalogRegisters(BOARD)
DR = DigitalRegisters(BOARD)
BC = get_board_controller(BOARD)
RC = get_readout_controller(BOARD)
DAC_CTRL = get_dac_controller(BOARD)
AM = AcquisitionManager(BOARD)

DAC_CTRL.set_dacs(DAC_LEVEL)
PG = get_pedestals_controller(BOARD, num_captures=10, num_warmup_events=10)

READ_WINDOW = {
    "windows": WINDOWS,
    "lookback": LOOKBACK,
    "write_after_trig": WAT,
}
READOUT_PARAMS = {
    "trig": "ext",
    "lb": "trigrel",
}
DR.write("wlcon", 0)
BOARD.initialize()
PG.generate_pedestals()
RC.set_read_window(**READ_WINDOW)
RC.set_readout_channels(READOUT_CHANNELS)
DR.write("wlcon", WLC_ON)
params = {
    "dac_level": DAC_LEVEL,
    "windows": WINDOWS,
    "lookback": LOOKBACK,
    "write_after_trig": WAT,
    "trigger_time": TRIGGER_TIME,
    "n_cycle": N_CYCLES,
    "burst_period": BURST_PERIOD,
    "pulse_width": PULSE_WIDTH,
    "pulse_freq": PULSE_FREQ,
    "wlc_on": WLC_ON,
    "readout_channels": READOUT_CHANNELS,
}
readout_metadata = {"params": deepcopy(params)}
RC.set_read_window(**READ_WINDOW)

try:
    ACQ_NAME = f"{BOARD.model}_windows{WINDOWS}_ncyc{N_CYCLES}_wlc{WLC_ON}"
    acq_name = create_acq_name(ACQ_NAME)
    acq = create_acquisition(acq_name, BOARD, readout_metadata)
    acq.pedestals = BOARD.pedestals
    BC.start_readout(**READOUT_PARAMS)
    time.sleep(0.1)
    CHANNEL_INPUT.rf_on()
    TRIGGER_INPUT.rf_on()
    for _ in range(TRIGGER_TIME):
        time.sleep(1)
finally:
    BC.stop_readout()
    TRIGGER_INPUT.rf_off()
    CHANNEL_INPUT.rf_off()
    time.sleep(0.5)
    print(acq.length)
    AM.current_acquisition = None


In [ ]:
# If you want to turn WLC off, set `wlcon` to 0, then re-initialized the board
# DR.write("wlcon", 0)
# BOARD.initialize()

In [ ]:
PARSER = get_parser(BOARD.params)

# acq = AM.list()[0] # Get the first named acq from the backend
print(acq.readout_metadata)
event_num = 0
parsed = acq.parsed_event(PARSER, index=event_num)

fig = create_fig((10, 6))
plot_single_event(parsed, show=True)
plt.savefig(FIGURES_DIR / f"{acq.name}_event{event_num}.png")